## Local IndicTrans2 Finetuning (Auto-Resume Enabled)

This notebook is configured for your **Local Environment (Single GPU)**.

**Prerequisites:**
- Ensure you have selected your project's `.venv` as the Jupyter kernel.
- Configure your `.env` file first based on `.env.example`.

### How it works:
- **Environment**: Automatically loads secrets from `.env`.
- **Checkpointing**: Every 500 steps, a model checkpoint is saved and pushed to Hugging Face.
- **Resuming**: Downloads the last checkpoint from Hugging Face for seamless resumption.

In [4]:
# Cell 1: Setup Workspace & Load Variables from .env
import os

# Move to project root if running from inside notebooks/
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print("Current Working Directory:", os.getcwd())

# Simple .env parser to avoid extra dependency just for notebook
if os.path.exists(".env"):
    with open(".env") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, val = line.split("=", 1)
                os.environ[key.strip()] = val.strip().strip("'\"")
    print("Loaded environment variables from .env")
else:
    print("WARNING: No .env file found. Please create one based on .env.example")

Current Working Directory: /home/firojpaudel/Translation_en_indic_finetuning
Loaded environment variables from .env


In [ ]:
# Cell 2: Authenticate with Weights & Biases and Hugging Face
import wandb
from huggingface_hub import login

# Automatically uses tokens from .env
wandb.login()

if "HF_TOKEN" in os.environ:
    login(token=os.environ["HF_TOKEN"])
else:
    print("WARNING: HF_TOKEN not found in environment.")

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:wand

In [ ]:
# Cell 3: Auto-Resume Preparation
import os
os.makedirs("./checkpoints", exist_ok=True)

# Download the last checkpoint to resume if needed.
# Uses uv run to ensure it runs in the local .venv
!uv run hf download Firoj112/indictrans2-en-npi-mai-finetuned --local-dir ./checkpoints

In [ ]:
# Cell 4: Kick off Finetuning! (Single GPU setup)
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

!uv run accelerate launch --num_processes=1 --num_machines=1 --dynamo_backend=no --mixed_precision=fp16 \
    run_pipeline.py --config configs/default.yaml \
    --override hub.push_to_hub=true \
               hub.model_id=Firoj112/indictrans2-en-npi-mai-finetuned \
               training.mixed_precision=fp16